# 9. Hybrid Stacking Ensemble and Final Model Selection

## 9.1 Objective

The objective of this notebook is to programmatically select the top two optimized classifiers based on cross-validation performance, construct a Stacking Classifier hybrid ensemble, evaluate its generalizability across both synthetic and real test data slices, and objectively determine the final model selection based on empirical real-world validation.


In [1]:
import os
import sys
import time
import json
import pickle
import numpy as np
import pandas as pd
import scipy.sparse
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
import joblib

# Configure stdout encoding to utf-8
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

# Robust Project root setup
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "datasets" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "preprocessing"
BEST_MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)


Project root: D:\newwwwwwww\AiBasedInstagramPrediction
Processed data directory: D:\newwwwwwww\AiBasedInstagramPrediction\datasets\processed
Results directory: D:\newwwwwwww\AiBasedInstagramPrediction\results


## 9.2 Methodology

Our stacking ensemble uses the top two cross-validated models as base estimators and a simple `LogisticRegression(random_state=42)` as the meta-classifier. Stratified internal 5-fold cross-validation is used during training on the development split only to prevent target/vocabulary leakages from contaminating held-out test evaluations.


## 9.3 Load Preprocessed Data

We load the preprocessed training/testing matrices, labels, and tracking masks from the processed datasets directory.


In [2]:
# Load preprocessed arrays and labels
X_train = scipy.sparse.load_npz(PROCESSED_DIR / "X_train.npz")
X_test = scipy.sparse.load_npz(PROCESSED_DIR / "X_test.npz")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")['target']
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv")['target']
real_mask_train = np.load(PROCESSED_DIR / "real_mask_train.npy")
real_mask_test = np.load(PROCESSED_DIR / "real_mask_test.npy")

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")


X_train shape: (81600, 1517), y_train shape: (81600,)
X_test shape: (20400, 1517), y_test shape: (20400,)


## 9.4 Load Optimisation Results and Select Top Two Models

We load the cross-validation reports from Notebook 08 and rank the candidates by `Mean_CV_F1` to automatically select the top two estimators.


In [3]:
# Load optimization CV results
cv_results = pd.read_csv(RESULTS_DIR / "model_optimization_cv_results.csv")
ranked_cv = cv_results.sort_values("Mean_CV_F1", ascending=False).reset_index(drop=True)

# Add Rank column
ranked_cv.insert(0, 'Rank', ranked_cv.index + 1)
print("Optimized models ranked by cross-validated Weighted F1:")
display(ranked_cv)

# Save ranked top models
ranked_cv.to_csv(RESULTS_DIR / "top_two_models.csv", index=False)
print("Saved top_two_models.csv")

top_one_name = ranked_cv.loc[0, 'Model']
top_two_name = ranked_cv.loc[1, 'Model']

print("\nTOP TWO MODELS SELECTED FOR HYBRID MODEL:")
print(f"1. {top_one_name} (Mean CV F1: {ranked_cv.loc[0, 'Mean_CV_F1']:.4f})")
print(f"2. {top_two_name} (Mean CV F1: {ranked_cv.loc[1, 'Mean_CV_F1']:.4f})")


Optimized models ranked by cross-validated Weighted F1:
   Rank                Model  Mean_CV_F1  CV_F1_Std
0     1  Logistic Regression    0.445799   0.003774
1     2           Linear SVM    0.443456   0.004182
2     3        Random Forest    0.429924   0.002383
3     4          Extra Trees    0.416123   0.001453
Saved top_two_models.csv

TOP TWO MODELS SELECTED FOR HYBRID MODEL:
1. Logistic Regression (Mean CV F1: 0.4458)
2. Linear SVM (Mean CV F1: 0.4435)


## 9.5 Stacking Classifier Construction

We build our Stacking Classifier base estimators dynamically from the top two selected classifiers, using their optimized parameters, and set Logistic Regression as the meta-learner.


In [4]:
# Helper function to load estimator with best parameters found in Notebook 08
def build_estimator(name):
    if name == "Logistic Regression":
        return LogisticRegression(C=0.01, solver='lbfgs', max_iter=1000, random_state=42)
    elif name == "Linear SVM":
        return LinearSVC(C=0.1, random_state=42, max_iter=2000)
    elif name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, min_samples_split=2, max_depth=20, random_state=42, n_jobs=-1)
    else:
        return ExtraTreesClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)

base_est_1 = build_estimator(top_one_name)
base_est_2 = build_estimator(top_two_name)

base_estimators = [
    (top_one_name.replace(" ", "_").lower(), base_est_1),
    (top_two_name.replace(" ", "_").lower(), base_est_2)
]

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=5,
    n_jobs=-1,
    stack_method='auto'
)

print(f"Constructed Stacking Classifier with base estimators: {top_one_name} and {top_two_name}")


Constructed Stacking Classifier with base estimators: Logistic Regression and Linear SVM


## 9.6 Stacking Model Training

We train the Stacking Classifier on the training split and evaluate the training duration.


In [5]:
print("Fitting stacking classifier on training split...")
t0 = time.time()
stacking_model.fit(X_train, y_train)
fit_time = time.time() - t0
print(f"Stacking Classifier fit complete in {fit_time:.2f}s")


Fitting stacking classifier on training split...
Stacking Classifier fit complete in 37.13s


## 9.7 Stacking Model Evaluation

We evaluate the stacking classifier, as well as the best and second-best individual models, on the combined test set to compare accuracy, precision, recall, and F1 metrics.


In [6]:
# Fit individual estimators on training data to compare predictions
indiv_1 = build_estimator(top_one_name)
indiv_2 = build_estimator(top_two_name)

print("Fitting individual model 1...")
indiv_1.fit(X_train, y_train)
print("Fitting individual model 2...")
indiv_2.fit(X_train, y_train)

# Predictions on X_test
pred_ind1 = indiv_1.predict(X_test)
pred_ind2 = indiv_2.predict(X_test)
pred_stack = stacking_model.predict(X_test)

# Metric evaluations
def get_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return round(acc, 4), round(prec, 4), round(rec, 4), round(f1, 4)

met_1 = get_metrics(y_test, pred_ind1)
met_2 = get_metrics(y_test, pred_ind2)
met_stack = get_metrics(y_test, pred_stack)

comparison_records = [
    {"Model": top_one_name, "Type": "Individual", "Accuracy": met_1[0], "Precision": met_1[1], "Recall": met_1[2], "Weighted_F1": met_1[3]},
    {"Model": top_two_name, "Type": "Individual", "Accuracy": met_2[0], "Precision": met_2[1], "Recall": met_2[2], "Weighted_F1": met_2[3]},
    {"Model": f"{top_one_name} + {top_two_name}", "Type": "Hybrid Stacking", "Accuracy": met_stack[0], "Precision": met_stack[1], "Recall": met_stack[2], "Weighted_F1": met_stack[3]}
]

comparison_df = pd.DataFrame(comparison_records)
print("Model Comparison on Combined Test Set:")
display(comparison_df)

comparison_df.to_csv(RESULTS_DIR / "hybrid_model_results.csv", index=False)
print("Saved hybrid_model_results.csv")


Fitting individual model 1...
Fitting individual model 2...
Model Comparison on Combined Test Set:
                              Model             Type  ...  Recall  Weighted_F1
0               Logistic Regression       Individual  ...  0.4495       0.4443
1                        Linear SVM       Individual  ...  0.4500       0.4420
2  Logistic Regression + Linear SVM  Hybrid Stacking  ...  0.4519       0.4464

[3 rows x 6 columns]
Saved hybrid_model_results.csv


## 9.8 Stacking Model Validation

We evaluate our final estimators separately on the held-out REAL dataset to capture performance outcomes under domain shifts.


In [7]:
# Slicing test set by real mask
y_test_real = y_test[real_mask_test]
pred_ind1_real = pred_ind1[real_mask_test]
pred_stack_real = pred_stack[real_mask_test]

# Slice test set by synthetic mask
y_test_synth = y_test[~real_mask_test]
pred_ind1_synth = pred_ind1[~real_mask_test]
pred_stack_synth = pred_stack[~real_mask_test]

# Get metrics
met_ind1_real = get_metrics(y_test_real, pred_ind1_real)
met_stack_real = get_metrics(y_test_real, pred_stack_real)

met_ind1_synth = get_metrics(y_test_synth, pred_ind1_synth)
met_stack_synth = get_metrics(y_test_synth, pred_stack_synth)

validation_records = [
    {"Dataset": "Real test observations only", "Model": top_one_name, "Accuracy": met_ind1_real[0], "Precision": met_ind1_real[1], "Recall": met_ind1_real[2], "Weighted_F1": met_ind1_real[3]},
    {"Dataset": "Real test observations only", "Model": "Hybrid Stacking Model", "Accuracy": met_stack_real[0], "Precision": met_stack_real[1], "Recall": met_stack_real[2], "Weighted_F1": met_stack_real[3]},
    {"Dataset": "Synthetic test observations only", "Model": top_one_name, "Accuracy": met_ind1_synth[0], "Precision": met_ind1_synth[1], "Recall": met_ind1_synth[2], "Weighted_F1": met_ind1_synth[3]},
    {"Dataset": "Synthetic test observations only", "Model": "Hybrid Stacking Model", "Accuracy": met_stack_synth[0], "Precision": met_stack_synth[1], "Recall": met_stack_synth[2], "Weighted_F1": met_stack_synth[3]}
]

validation_df = pd.DataFrame(validation_records)
print("Comparative Evaluation by Source Dataset:")
display(validation_df)

validation_df.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)
print("Saved final_model_comparison.csv")


Comparative Evaluation by Source Dataset:
                            Dataset                  Model  ...  Recall  Weighted_F1
0       Real test observations only    Logistic Regression  ...  0.4227       0.4232
1       Real test observations only  Hybrid Stacking Model  ...  0.4330       0.4364
2  Synthetic test observations only    Logistic Regression  ...  0.4500       0.4444
3  Synthetic test observations only  Hybrid Stacking Model  ...  0.4523       0.4464

[4 rows x 6 columns]
Saved final_model_comparison.csv


## 9.9 Stacking Model Validation Confusion Matrix

We save a professional confusion display of the final hybrid model predictions on the held-out real evaluation set.


In [8]:
class_labels = ["Low", "Medium", "High"]
cm_stack = confusion_matrix(y_test_real, pred_stack_real)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_stack, display_labels=class_labels)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title("Hybrid Stacking Model Confusion Matrix (Real Data)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "hybrid_confusion_matrix.png")
plt.close()
print("Saved hybrid_confusion_matrix.png")


Saved hybrid_confusion_matrix.png


## 9.10 Error Analysis

We profile class confusions for our hybrid stacking model, detailing low-medium-high misclassifications on real test observations.


In [9]:
misclass_real = y_test_real != pred_stack_real
print(f"Total misclassifications on Real data: {np.sum(misclass_real)} out of {len(y_test_real)} ({np.sum(misclass_real)/len(y_test_real)*100:.2f}%)")

# Confusions
low_as_med_r = np.sum((y_test_real == 0) & (pred_stack_real == 1))
low_as_high_r = np.sum((y_test_real == 0) & (pred_stack_real == 2))
med_as_low_r = np.sum((y_test_real == 1) & (pred_stack_real == 0))
med_as_high_r = np.sum((y_test_real == 1) & (pred_stack_real == 2))
high_as_low_r = np.sum((y_test_real == 2) & (pred_stack_real == 0))
high_as_med_r = np.sum((y_test_real == 2) & (pred_stack_real == 1))

print(f"  Low predicted as Medium: {low_as_med_r}")
print(f"  Low predicted as High: {low_as_high_r}")
print(f"  Medium predicted as Low: {med_as_low_r}")
print(f"  Medium predicted as High: {med_as_high_r}")
print(f"  High predicted as Low: {high_as_low_r}")
print(f"  High predicted as Medium: {high_as_med_r}")


Total misclassifications on Real data: 220 out of 388 (56.70%)
  Low predicted as Medium: 64
  Low predicted as High: 22
  Medium predicted as Low: 40
  Medium predicted as High: 23
  High predicted as Low: 33
  High predicted as Medium: 38


## 9.11 Final Model Selection & Interpretation

We objectively evaluate Stacking vs. Best Individual model performance on Real test data. If the hybrid stacking model demonstrates a superior Real F1-score, we select it as the final estimator. Otherwise, we retain the best individual model.


In [10]:
# Evaluate if hybrid stacking outperforms best individual model on Real test F1
hybrid_f1 = met_stack_real[3]
best_indiv_f1 = met_ind1_real[3]

print(f"Hybrid Stacking Real F1: {hybrid_f1:.4f}")
print(f"Best Individual ({top_one_name}) Real F1: {best_indiv_f1:.4f}")

if hybrid_f1 > best_indiv_f1:
    hybrid_improved = True
    final_selected_model = "Hybrid Stacking Model"
    final_estimator_obj = stacking_model
    final_f1 = hybrid_f1
    print("\nEmpirical Selection: Stacking Ensemble provides Complementary predictive signal.")
else:
    hybrid_improved = False
    final_selected_model = f"Best Individual Model ({top_one_name})"
    final_estimator_obj = indiv_1
    final_f1 = best_indiv_f1
    print("\nEmpirical Selection: Stacking Ensemble did NOT provide empirical improvement on real data. Retaining best individual model.")

print(f"Final Selected Model: {final_selected_model}")


Hybrid Stacking Real F1: 0.4364
Best Individual (Logistic Regression) Real F1: 0.4232

Empirical Selection: Stacking Ensemble provides Complementary predictive signal.
Final Selected Model: Hybrid Stacking Model


## 9.12 Save Hybrid Artifacts

We save the final selected model using joblib and export its parameters and meta-information.


In [11]:
# Save best model to models
joblib.dump(final_estimator_obj, BEST_MODEL_DIR / "best_engagement_model.joblib")
print("Saved best_engagement_model.joblib")

# Save final parameters metadata
final_params = {
    "selected_model": final_selected_model,
    "hybrid_improved_performance": hybrid_improved,
    "top_one_base_estimator": top_one_name,
    "top_two_base_estimator": top_two_name,
    "meta_classifier": "LogisticRegression(random_state=42)",
    "real_data_accuracy": met_stack_real[0] if hybrid_improved else met_ind1_real[0],
    "real_data_weighted_f1": float(final_f1)
}

with open(RESULTS_DIR / "final_model_parameters.json", "w") as f:
    json.dump(final_params, f, indent=4)
print("Saved final_model_parameters.json")


Saved best_engagement_model.joblib
Saved final_model_parameters.json


## 9.13 Academic Summary

The hybrid ensemble and selection stage is completed. Below is the programmatic summary of the results:


In [12]:
summary_info = {
    "Top_Two_Selected_Models": [top_one_name, top_two_name],
    "Why_Selected": "Selected objectively based on cross-validated Weighted F1 scores",
    "Hybrid_Architecture": "StackingClassifier with LogisticRegression meta-classifier",
    "Hybrid_Performance_Real_F1": met_stack_real[3],
    "Individual_Performance_Real_F1": met_ind1_real[3],
    "Real_Data_Performance_F1": float(final_f1),
    "Whether_Hybridisation_Improved_Performance": "Yes" if hybrid_improved else "No",
    "Final_Selected_Model": final_selected_model,
    "Remaining_Limitations": "Domain shift between synthetic training features and real-world scraping records remains present. Modality gaps in visual characteristics for real images limit joint prediction capabilities."
}

print(json.dumps(summary_info, indent=4))

print("\nHIBRID ENSEMBLE COMPLETED")
print("NEXT STEP: READY FOR FINAL MODEL SELECTION AND IMAGE-ENHANCED EXPERIMENT.")


{
    "Top_Two_Selected_Models": [
        "Logistic Regression",
        "Linear SVM"
    ],
    "Why_Selected": "Selected objectively based on cross-validated Weighted F1 scores",
    "Hybrid_Architecture": "StackingClassifier with LogisticRegression meta-classifier",
    "Hybrid_Performance_Real_F1": 0.4364,
    "Individual_Performance_Real_F1": 0.4232,
    "Real_Data_Performance_F1": 0.4364,
    "Whether_Hybridisation_Improved_Performance": "Yes",
    "Final_Selected_Model": "Hybrid Stacking Model",
    "Remaining_Limitations": "Domain shift between synthetic training features and real-world scraping records remains present. Modality gaps in visual characteristics for real images limit joint prediction capabilities."
}

HIBRID ENSEMBLE COMPLETED
NEXT STEP: READY FOR FINAL MODEL SELECTION AND IMAGE-ENHANCED EXPERIMENT.
